In [2]:
import os
from osgeo import gdal
import geopandas as gpd
import numpy as np
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import math

In [3]:
def get_file_list(path):
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file not in File_list:
            File_list.append(os.path.join(path,file))
    return File_list

def get_unique_values_chunked(raster_path, chunk_size=1024, exclude_nodata=True):
    """
    Return unique pixel values of a raster band by reading in chunks.

    Parameters
    ----------
    raster_path: str, Path to raster file.
    chunk_size: int, optional Number of rows/cols per chunk to read at once (default: 1024).
    exclude_nodata: bool, optional Exclude NoData value from result (default: True).
    """
    ds = gdal.Open(raster_path)
    if ds is None:
        raise FileNotFoundError(f"Cannot open raster: {raster_path}")

    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()

    xsize = band.XSize
    ysize = band.YSize

    # Calculate how many chunks will be processed
    n_chunks_x = (xsize + chunk_size - 1) // chunk_size
    n_chunks_y = (ysize + chunk_size - 1) // chunk_size
    total_chunks = n_chunks_x * n_chunks_y

    unique_values = set()
    chunk_counter = 0

    print(f"Processing {total_chunks} chunks...")

    for y in range(0, ysize, chunk_size):
        rows = min(chunk_size, ysize - y)
        for x in range(0, xsize, chunk_size):
            cols = min(chunk_size, xsize - x)

            data = band.ReadAsArray(x, y, cols, rows)
            if data is None:
                continue

            if exclude_nodata and nodata is not None:
                data = data[data != nodata]

            unique_values.update(np.unique(data))

            # Update progress
            chunk_counter += 1
            if chunk_counter % 10 == 0 or chunk_counter == total_chunks:
                percent = (chunk_counter / total_chunks) * 100
                print(f"  Chunk {chunk_counter}/{total_chunks} ({percent:.1f}%)", end="\r", flush=True)

    ds = None  # close dataset
    return sorted(unique_values)

def get_vector_unique_values(file_path, col_name_1 = None, col_name_2 = None):

    gdf = gpd.read_file(file_path)

    unique_values = {}
    for col in gdf.columns:
        print(f"we are in column: {col}", end = "\n")
        # Skip geometry column
        if col != gdf.geometry.name:
            unique_values[col] = gdf[col].dropna().unique().tolist()

    # ---- ALIGN TO SAME LENGTH ----
    max_len = max(len(v) for v in unique_values.values())
    for col in unique_values:
        unique_values[col] += [""] * (max_len - len(unique_values[col]))

    # ---- CREATE NEW DATAFRAME ----
    unique_df = pd.DataFrame(unique_values)

    if col_name_1 and col_name_2:
    # Get the combinations between two columns
        unique_combo_df = gdf[[ col_name_1, col_name_2 ]].drop_duplicates()
        
        return unique_df, unique_combo_df

    return unique_df

def classify_format_file(path):
    vector_exts = {".shp", ".gpkg"}
    raster_exts = {".tif", ".tiff"}

    # Get the extension
    ext = os.path.splitext(path)[1].lower()

    if ext in vector_exts:
        return "vector"
    elif ext in raster_exts:
        return "raster"
    else:
        return "unknown"

def get_wcs_layers(base_url):
    params = {
        "service": "WCS",
        "version": "2.0.1",
        "request": "GetCapabilities"
    }

    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    # Parse XML
    root = ET.fromstring(response.content)

    # WCS 2.0 namespace map
    ns = {
        "wcs": "http://www.opengis.net/wcs/2.0",
        "ows": "http://www.opengis.net/ows/2.0"
    }

    layers = []

    # Find all CoverageId elements
    for coverage in root.findall(".//wcs:CoverageSummary", ns):
        coverage_id = coverage.find("wcs:CoverageId", ns)
        if coverage_id is not None:
            layers.append(coverage_id.text)

    return layers

def wcs_describe_coverage_bbox(base_url, coverage_id, version):
    """
    base_url = "geoserver/url"
    coverage_id = "workspace:layername"
    version = "1.0.0" or "2.0.1"

    Returns:
        xmin, ymin, xmax, ymax, width, height   (in native CRS)
    """


    # --- Choose correct coverage parameter name ---
    if version == "2.0.1":
        coverage_param = "CoverageId"
    elif version == "1.0.0":
        coverage_param = "coverage"
    else:
        raise ValueError("Unsupported WCS version")

    params = {
        "service": "WCS",
        "version": version,
        "request": "DescribeCoverage",
        coverage_param: coverage_id
    }

    r = requests.get(base_url, params=params, timeout=60)
    print(r.url)
    r.raise_for_status()

    root = ET.fromstring(r.content)

    # --- Namespaces depend on WCS version ---
    if version == "2.0.1":
        ns = {
            "wcs": "http://www.opengis.net/wcs/2.0",
            "gml": "http://www.opengis.net/gml/3.2"
        }
    else:  # 1.0.0
        ns = {
            "wcs": "http://www.opengis.net/wcs",
            "gml": "http://www.opengis.net/gml"
        }

    # ==========================================================
    # Parse envelope (native CRS, not lat/lon one)
    # ==========================================================

    envelope = root.find(".//gml:Envelope", ns)
    if envelope is None:
        raise RuntimeError("Could not find gml:Envelope in DescribeCoverage response")

    if version == "2.0.1":
        lower = envelope.find("gml:lowerCorner", ns)
        upper = envelope.find("gml:upperCorner", ns)
        if lower is None or upper is None:
            raise RuntimeError("Could not find lowerCorner/upperCorner in Envelope")

        p1 = list(map(float, lower.text.strip().split()))
        p2 = list(map(float, upper.text.strip().split()))
        
        # Get the axis labels
        axis_labels = envelope.get("axisLabels")
        
        # Get the second axis labels
        grid = root.find(".//gml:RectifiedGrid", ns)
        second_axis_element = grid.find("gml:axisLabels", ns)
        second_axis_labels = second_axis_element.text.strip().split()
        
        if axis_labels:
            axis_labels = axis_labels.split()
        else:
            print("No axisLabels found in Envelope")
            
        if not second_axis_labels:
            print("No gml:axisLabels found in Envelope")

    else:  # 1.0.0 (uses gml:pos)
        positions = envelope.findall("gml:pos", ns)
        if len(positions) != 2:
            raise RuntimeError("Unexpected Envelope structure (expected 2 gml:pos elements)")

        p1 = list(map(float, positions[0].text.strip().split()))
        p2 = list(map(float, positions[1].text.strip().split()))

    xmin = min(p1[0], p2[0])
    ymin = min(p1[1], p2[1])
    xmax = max(p1[0], p2[0])
    ymax = max(p1[1], p2[1])

    # ==========================================================
    # Parse grid size
    # ==========================================================

    # WCS 2.0 path is slightly different
    grid = root.find(".//gml:RectifiedGrid//gml:GridEnvelope", ns)
    if grid is None:
        grid = root.find(".//gml:RectifiedGrid//gml:limits//gml:GridEnvelope", ns)

    if grid is None:
        raise RuntimeError("Could not find gml:GridEnvelope in DescribeCoverage response")

    low_elem = grid.find("gml:low", ns)
    high_elem = grid.find("gml:high", ns)

    if low_elem is None or high_elem is None:
        raise RuntimeError("GridEnvelope missing gml:low or gml:high")

    low = list(map(int, low_elem.text.strip().split()))
    high = list(map(int, high_elem.text.strip().split()))

    width  = high[0] - low[0] + 1
    height = high[1] - low[1] + 1
    
    if version == "2.0.1":
        properties = [ymin, xmin, ymax, xmax, width, height, axis_labels, second_axis_labels]
        return properties
    
    else:
        properties = [ymin, xmin, ymax, xmax, width, height]
    return properties

def wcs_getcoverage(base_url, coverage_id, properties, output_file, version, simple_coverage = False):

    if version == "2.0.1":
        ymin, xmin, ymax, xmax, width, height, axes, second_axes = properties
    else: # version == "1.0.0":
        ymin, xmin, ymax, xmax, width, height = properties

    if simple_coverage is True:
        max_pixels = 10000000
        total_pixels = width * height

        if total_pixels > max_pixels:
            scale = math.sqrt(max_pixels / total_pixels)

            width = math.ceil(width * scale) # ceil rounds up the value
            height = math.ceil(height * scale)

    crs = "EPSG:4326"

    if version == "2.0.1":

        # WCS 2.0.1 syntax

        # subset repeated per axis
        params = [
            ("service", "WCS"),
            ("version", "2.0.1"),
            ("request", "GetCoverage"),
            ("CoverageId", coverage_id),
            ("format", "image/geotiff"),

            ("subset", f"{axes[0]}({xmin},{xmax})"),
            ("subset", f"{axes[1]}({ymin},{ymax})")
            
            # ("ScaleSize", f"{second_axes[0]}({width}),{second_axes[1]}({height})")
        ]
        
        # Manually construct the URL to append ScaleSize exactly
        req = requests.Request("GET", base_url, params=params).prepare()
        url = f"{req.url}&ScaleSize={second_axes[0]}({width}),{second_axes[1]}({height})"
        

    elif version == "1.0.0":

        # WCS 1.0.0 syntax
        url = (
            f"{base_url}?service=WCS&version=1.0.0&request=GetCoverage&coverage={coverage_id}"
            f"&crs={crs}&bbox={ymin},{xmin},{ymax},{xmax}&width={width}&height={height}&format=GeoTIFF"
        )
    else:
        raise ValueError("Unsupported WCS version")

    try:
        
        # print("WCS request URL:")
        # print(req.url)

        s = requests.Session()
        r = s.get(url, timeout=120)

        r.raise_for_status()

        with open(output_file, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)

        return output_file

    except Exception as e:
        print("WCS request failed:")
        print(e)
        return None

In [6]:
"""OPTION: Extract all the used layers"""
base_url = "https://integration.integratedmodelling.org/geoserver/ows"
coverage_id = "im-data-global-ecology__europe_primary_forests_100m_4326_2018"
version = "2.0.1"

# take the layer name as an output
output_file = rf".\{coverage_id.split('__', 1)[1]}.tif"

# Get the properties of the layer
properties = wcs_describe_coverage_bbox(base_url, coverage_id, version)

ymin, xmin, ymax, xmax, width, height, *_ = properties # *_ ignore the rest

# Get the layer
file_path = wcs_getcoverage(base_url, coverage_id, properties, output_file, version, simple_coverage=False)


https://integration.integratedmodelling.org/geoserver/ows?service=WCS&version=2.0.1&request=DescribeCoverage&CoverageId=im-data-global-ecology__europe_primary_forests_100m_4326_2018
WCS request failed:
HTTPSConnectionPool(host='integration.integratedmodelling.org', port=443): Read timed out. (read timeout=120)


In [ ]:
"""Inputs"""
main_parth = r"main_path"
# For raster if there is a 
input_csv = r"whatever.csv"

output_path = r"main_path.csv"

# For vectors files, the column pair
# Columns to 
col_name_1 = ""
col_name_2 = ""

#klab
klab_data_tracker = 5

In [ ]:
files_list = get_file_list(main_parth)
# get_unique_values_chunked(raster_path, chunk_size=1024, exclude_nodata=True)

In [ ]:
accumulative_df = []

for file_path in files_list[:]:
    file_format = classify_format_file(file_path)
    if file_format == "raster":
        unique_values = get_unique_values_chunked(file_path, chunk_size=1024, exclude_nodata=True)

        # This is to create a table with all the text, unique values.
        raster_name = os.path.basename(file_path).replace(".tif","")
        df = pd.DataFrame(sorted(unique_values), columns=[raster_name])
        accumulative_df.append(df)
        cumulative_df = pd.concat(accumulative_df, axis=1)

    elif file_format == "vector":
        
        unique_values_df, unique_combo_df = get_vector_unique_values(file_path)
        # I don't find unique_values_df super useful, so I set it like that

        # Get the vector name
        # vector_name = os.path.splitext(os.path.basename(file_path))[0]
        accumulative_df.append(unique_combo_df)
        cumulative_df = pd.concat(accumulative_df, axis=1)


# ---- SAVE TO CSV ----
accumulative_df.to_csv(output_path, index=False)

print(f"✅ Unique values saved to: {output_path}")

        # Optional